# Catalog

Use the **bao** kernel in your allocated HPC remote desktop. Edit these cells and the reusable functions in `pzbao/diagnostics/catalog.py`.

These small synthetic examples illustrate the interface; they are not BAO results. Scientific interpretation and acceptance remain yours.


In [ ]:
from pathlib import Path
import numpy as np
from pzbao.diagnostics import catalog as diagnostics
from pzbao.config import load_analysis_config
from pzbao.config.stages import stage_document
from pzbao.run import load_run, stage_units

RUN_ROOT = None  # Set to the named run directory you want to inspect.
run = load_run(RUN_ROOT) if RUN_ROOT is not None else None
config = run["config"] if run is not None else load_analysis_config()


## Settings and saved products

This is the section used by `bao catalog`. Edit a complete stage YAML or a combined experiment YAML. CLI runs freeze scientific settings; use a new named run for a changed experiment.


In [ ]:
settings = stage_document(config, "catalog")
settings


In [ ]:
if run is not None:
    units = stage_units("catalog", config, run["metadata"]["workload"])
    products = list((run["root"] / "catalog" / "units").glob("*/result.yaml"))
    print(f"{len(products)} saved units; {len(units)} configured units")


## Explore and plot

Replace the illustrative arrays with saved arrays from your chosen unit. Plotting functions accept an existing axis and return their data for inspection.


In [ ]:
sample = {"SOURCE_ID": np.arange(12, dtype=np.uint64), "Z": np.linspace(0.4, 1.0, 12)}
summary = diagnostics.describe(sample)
ax, values = diagnostics.plot_redshift(sample, bins=6)
summary


## Load one saved result

Set `LOAD_SAVED = True` after selecting your run and unit. These cells only read saved products; missing prerequisites remain visible as errors. The clustering example reads xi; P(k)-only runs can be inspected through the unit's `measurements.json` and named power NPZ files.


In [ ]:
LOAD_SAVED = False
SAVED_UNIT_INDEX = 0  # Choose the configured unit/route to inspect.
if LOAD_SAVED:
    if run is None:
        raise ValueError("Set RUN_ROOT before loading saved products")
    from pzbao.catalog import read_catalog_registration, read_registered_catalogue

    registration = read_catalog_registration(run, units[SAVED_UNIT_INDEX]["tracer"])
    source = next(
        row for row in registration["sources"] if row["role"] == "data" and row["cap"] == "NGC"
    )
    saved = read_registered_catalogue(
        source,
        tracer=source["tracer"],
        mock_id=source["mock_id"],
        cap=source["cap"],
        role=source["role"],
        config=config,
    )
    saved_ax, saved_values = diagnostics.plot_redshift(saved, label="Registered source")


## Interpretation

Record the chosen unit, comparisons, missing or failed products, relevant flags, and your scientific question here. Numerical stopping and report publication do not establish a validation verdict.
